# MHM and MsHHO field equivalence

This notebook compares the finite conforming Galerkin realization of moment-based MsHHO and primal MHM with a polynomial source. Exact-local-space equivalence is a literature result; the local $P_3$ realization here is a finite Galerkin analogue. General reconstructed sources and the face-only $m=-1$ formulation have separate mathematical conventions.


The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/792fed9148995db8fd9811e718870d4e7a7579145ae9f9284b2160e405c8b888/24_mshho-companion.zip"
COMPANION_SHA256 = "792fed9148995db8fd9811e718870d4e7a7579145ae9f9284b2160e405c8b888"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/24_mshho.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


## Declare cell and face moment equations

The application writes integral functionals J and obtains the constrained
energy reconstruction R from the shared numerical kernel. Its Galerkin energy
operator is split into the actual four local/global blocks:

$$
\begin{aligned}
E_T&=R_T^{\mathsf T}K_TR_T, & J_T^{\mathsf T}R_T&=I,\\
A_T&=E_{cc}, &B_T&=E_{cf}, &C_T&=E_{fc}, &D_T&=E_{ff}.
\end{aligned}
$$

The local source functional follows the selected projected or reconstructed
variant. Generic assembly eliminates cell moments and sums face equations.
Dirichlet data fix face integrals; the global `Equation` supplies outward
Neumann data in dual moment coordinates. Pressure recovery uses the exact
executed R matrix and preserves distinct local fields.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.moments import define_moment_diffusion, recover_moment_diffusion
from examples.formulations.moments import pressure_constraints as moment_pressure_constraints

from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy
import numpy as np
from pymhm import TriangleMesh, FaceSpace, SkeletonSpace

mesh = TriangleMesh.unit_square()
skeleton = SkeletonSpace(mesh, tuple((FaceSpace.uniform(1) for _ in mesh.faces)))
options = dict(skeleton=skeleton, degree=3, local_refinement=2, source=2.0)
mhm_definition = define_darcy(mesh, **options)
mhm_system = assemble(mhm_definition.problem)
mhm = recover_darcy(
    mhm_definition,
    mhm_system,
    mhm_system.solve(constraints=pressure_constraints(mhm_definition, mhm_system)),
)
hho_definition = define_moment_diffusion(mesh, cell_degree=1, **options)
hho_system = assemble(hho_definition.problem)
hho = recover_moment_diffusion(
    hho_definition,
    hho_system,
    hho_system.solve(constraints=moment_pressure_constraints(hho_definition, hho_system)),
)
error = max((np.max(np.abs(a - b)) for a, b in zip(mhm.pressure, hho.pressure, strict=True)))
assert error < 1e-11
error


## Face-only moment variant

For m=-1 the reconstructed-source variant is required. There are no cell moment
unknowns, so A is 0×0 and B/C have empty local dimensions; the declared D and g
carry the face equation. The generic core assembles these forms directly.
This additional affine analytical control uses the same P3 local fields and P1
face moments; it is separate from the preceding nonzero-source equivalence test.


In [ ]:
face_only_definition = define_moment_diffusion(
    mesh,
    skeleton=skeleton,
    cell_degree=-1,
    degree=3,
    local_refinement=2,
    source_variant="reconstructed",
    dirichlet=lambda x: 1 + x[:, 0] + 2 * x[:, 1],
)
face_only_system = assemble(face_only_definition.problem)
face_only = recover_moment_diffusion(
    face_only_definition,
    face_only_system,
    face_only_system.solve(
        constraints=moment_pressure_constraints(face_only_definition, face_only_system)
    ),
)
assert all(response.problem.matrix.shape == (0, 0) for response in face_only_system.responses)
assert all(moment.shape == (0,) for moment in face_only.cell_moments)
pressure_error = face_only.l2_error(lambda x: 1 + x[:, 0] + 2 * x[:, 1])
flux_error = face_only.flux_l2_error([-1.0, -2.0])
assert pressure_error < 1e-11 and flux_error < 1e-10
{"cell_degree": -1, "pressure_L2": pressure_error, "flux_L2": flux_error}


The complete five-level analytical and four-contrast campaign is acquired separately. The following cells display its current numerical record and figures.
The anisotropic comparison uses explicit wider storage and two original-equation corrections, with the original residual criterion $10^{-10}$. The independently assembled reference uses the same full eight-macrocell physical case. Physical field agreement and insertion into the other rounded operator are distinct checks; at contrast $10^6$ the latter remains outside its declared criterion.
The compact native-verification record identifies the executed Basix modules, source revision and actual bases. The comparison independently assembles its operator and shares the checked SciPy arithmetic.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    from pathlib import Path
    import json
    from IPython.display import Image, display
    record = json.loads((root / "examples/results/mshho.json").read_text())
    assert record["status"] == "verified-gallery"
    assert record["mhm_hybrid_refinement_min_steps"] == 2
    for row in record["equivalence"]:
        assert row["status"] == "verified"
        assert max(row["original_saddle_diagnostics"][method + "_original_relative_residual"] for method in ("mhm", "mshho")) <= 1e-10
    native = json.loads((root / "examples/results/mshho/native-verification.json").read_text())
    assert native["criteria"]["changed"] is False
    assert native["cross_operator_limit"]["cross_insertion_passed"] is False
    assert native["replay"]["all_sampled_fields_and_original_rows_bitwise_equal"] is True
    dict(scope=record["scope"], convergence=record["convergence"], equivalence=record["equivalence"], native_reference=native["reference_project"], independent_cross_operator_limit=native["cross_operator_limit"])


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/mshho/convergence.png")))
    display(Image(filename=str(root / "docs/figures/mshho/fields.png")))
